# MimicKit on Kaggle

Clone -> install -> train, using the **Newton** engine (Isaac Gym cannot be installed on Kaggle).

**Before running:** Notebook settings -> Accelerator = **GPU T4 x2** (or P100), Internet = **On**.
Also attach the MimicKit data pack dataset (see `docs/README_Kaggle.md` for how to create it).

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

## 2. Clone the repo

For a private repo, put a GitHub token in Kaggle Secrets under the name `GITHUB_TOKEN` and
use the commented-out clone line instead.

In [ ]:
import os

REPO_URL = "https://github.com/phanhieeus/MimicKit-intern.git"
REPO_DIR = "/kaggle/working/MimicKit-intern"

# Private repo:
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret("GITHUB_TOKEN")
# REPO_URL = f"https://{token}@github.com/phanhieeus/MimicKit-intern.git"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 $REPO_URL $REPO_DIR

os.chdir(REPO_DIR)          # every path in the configs is relative to the repo root
print(os.getcwd())
!ls

## 3. Install dependencies

Takes ~3-4 minutes. Kaggle's preinstalled CUDA build of torch is left untouched.

In [ ]:
!bash kaggle/setup.sh

## 4. Link the data pack into `data/`

Assets, motions and pretrained models are not in git. The script finds the attached
dataset under `/kaggle/input` and symlinks it into `data/`.

In [ ]:
!ls /kaggle/input
!python kaggle/prepare_data.py

## 5. Smoke test

Plays back a motion clip with the dummy agent for one episode. This is the cheapest way to
confirm that Newton, MuJoCo-Warp and the data pack all work together.

Note the CLI flags come **before** `--arg_file` values in precedence: MimicKit keeps the
first value it sees for a key, so anything passed on the command line overrides the arg file.
That is how `--engine_config data/engines/newton_engine.yaml` replaces the Isaac Gym default.

In [ ]:
!python mimickit/run.py \
    --arg_file args/view_motion_humanoid_args.txt \
    --engine_config data/engines/newton_engine.yaml \
    --num_envs 4 \
    --mode test \
    --test_episodes 1 \
    --visualize false \
    --devices cuda:0

## 6. Evaluate a pretrained model (optional)

The released checkpoints were trained in Isaac Gym, so the return under Newton is only
indicative -- this checks the full agent path end to end.

In [ ]:
!python mimickit/run.py \
    --arg_file args/deepmimic_humanoid_ppo_args.txt \
    --engine_config data/engines/newton_engine.yaml \
    --num_envs 16 \
    --mode test \
    --test_episodes 8 \
    --visualize false \
    --model_file data/models/deepmimic_humanoid_spinkick_model.pt \
    --devices cuda:0

## 7. Train

`num_envs` is lowered from the default 4096 to fit a 16 GB T4 -- raise it if memory allows.
`max_samples` bounds the run so it finishes inside Kaggle's session limit; drop it for an
open-ended run. Output goes to `/kaggle/working/output/`, which persists as notebook output.

Run this in the background (`nohup`) if you want to keep using other cells while it trains.

In [ ]:
OUT_DIR = "/kaggle/working/output/deepmimic_humanoid"

!python mimickit/run.py \
    --arg_file args/deepmimic_humanoid_ppo_args.txt \
    --engine_config data/engines/newton_engine.yaml \
    --mode train \
    --num_envs 1024 \
    --max_samples 200000000 \
    --visualize false \
    --video false \
    --logger tb \
    --out_dir $OUT_DIR \
    --devices cuda:0

## 8. Inspect the training log

In [ ]:
!ls -la /kaggle/working/output/deepmimic_humanoid
!tail -30 /kaggle/working/output/deepmimic_humanoid/log.txt

## 9. Record a video (optional, may fail headless)

Video recording drives Newton's OpenGL viewer through Xvfb. Kaggle's GPU containers do not
always expose a usable GL/EGL context, so treat this as best-effort -- if it errors, keep
training with `--video false`.

In [ ]:
!xvfb-run -a -s "-screen 0 1024x768x24" python mimickit/run.py \
    --arg_file args/deepmimic_humanoid_ppo_args.txt \
    --engine_config data/engines/newton_engine.yaml \
    --num_envs 4 \
    --mode test \
    --test_episodes 2 \
    --visualize false \
    --video true \
    --logger tb \
    --model_file data/models/deepmimic_humanoid_spinkick_model.pt \
    --out_dir /kaggle/working/output/video \
    --devices cuda:0